# TinyChirp SincNet-Chunked (Axon-ready)

Single end-to-end SincNet model designed to fit the Axon NPU constraints (filter dims ≤ 16, strides ≤ 31, pool kernels ≤ 32, tensor dims ≤ 1024) and reproduce a clean "two parts at execution" structure:

1. **Per-chunk frontend** — SincNet convolution on each 1024-sample chunk (stride 8), followed by a cascaded avg-pool down to **one value per channel per chunk**.
2. **Head classifier** — Conv2D + cascaded global pool + dense over the sequence of chunk features.

Audio stays at 16 kHz (full 0–8 kHz band visible to the sinc filters). Total audio length is `46 × 1024 = 47104` samples (~2.94 s, fits inside the existing `TARGET_AUDIO_LEN_TIME`).

Self-contained: a local `SincnetConvW` is defined inline so the existing `model_parts.SincnetConv` (height-axis) is not touched.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import math
import numpy as np
import tensorflow as tf

from utils import (
    TARGET_AUDIO_LEN_TIME,
    SAMPLE_RATE,
    SEED,
    DATASET_ROOT,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
    time_to_features,
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "sincnet_chunked"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
BATCH_SIZE = 32

## Chunked audio geometry

Input is shaped as `(NUM_CHUNKS, CHUNK_SIZE, 1)` NHWC: chunks live on the H axis, raw samples within a chunk live on W. A `SincnetConvW` kernel of shape `(1, k)` slides only along W, so each chunk row is processed independently with the same sinc filterbank — matches the deployment-time "frontend runs per chunk" semantics.

In [ ]:
CHUNK_SIZE = 1024              # 64 ms @ 16 kHz
NUM_CHUNKS = 46                # 46 * 1024 = 47104 samples ~ 2.94 s
TOTAL_LEN = NUM_CHUNKS * CHUNK_SIZE

# --- Stage 1: per-chunk frontend (sinc + small conv to mimic deeper sinc) ---
# Axon Conv2D filter dim cap = 16; SincNet wants odd kernel for symmetric filters.
SINCNET_FILTERS = 32
SINCNET_KERNEL = 15
SINCNET_STRIDE = 8

# "Mimic deeper sinc": small standard conv composes the filterbank into
# CHUNK_CONV_FILTERS learned features per time-frame.
# Constraint: 46 chunks * CHUNK_CONV_FILTERS features per chunk <= 1024.
CHUNK_CONV_FILTERS = 20
CHUNK_CONV_KERNEL = 5

# Per-chunk pool cascade: 127 -> 7 -> 1   (kernels <= 32)
# MaxPool — bird calls are transient; peaks survive ReLU better than averages.
CHUNK_POOL_1 = 16
CHUNK_POOL_2 = 7

# --- Stage 2: head over chunks (hierarchical, mimics sincnet_multi) ---
HEAD_CONV_1_FILTERS = 32
HEAD_CONV_2_FILTERS = 32
HEAD_CONV_KERNEL = 5
HEAD_POOL_1 = 2                # 46 -> 23
HEAD_POOL_2 = 23               # 23 -> 1

DENSE_HIDDEN = 64

In [ ]:
# Load WAVs directly (bypassing make_audio_datasets) so we don't need to pass
# sampling_rate= — that kwarg pulls in tensorflow-io, which is ABI-broken
# against TF 2.19. All clips in this dataset are already 16 kHz, so no resample.
train_raw = tf.keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "training",
    labels="inferred",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)
val_raw = tf.keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "validation",
    labels="inferred",
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_raw = tf.keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "testing",
    labels="inferred",
    batch_size=BATCH_SIZE,
    shuffle=False,
)
label_names = np.array(train_raw.class_names)

train_ds = train_raw.map(time_to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
val_ds   = val_raw.map(time_to_features,   num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
test_ds  = test_raw.map(time_to_features,  num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)

num_labels = len(label_names)
print("Classes:", label_names)
for x, y in train_ds.take(1):
    print("Dataset shape:", x.shape, "  labels:", y.shape)
print(f"Will crop {TARGET_AUDIO_LEN_TIME - TOTAL_LEN} samples from tail "
      f"and reshape to (NUM_CHUNKS={NUM_CHUNKS}, CHUNK_SIZE={CHUNK_SIZE}, 1)")

## SincNet layer with kernel on the **width** axis

Imported from `model_parts.SincnetConvW`. Filter shape is `(1, kernel_size, 1, num_filters)` and the conv strides along W, so each H row (chunk) gets the same learnable bandpass filterbank — same kernel, processed independently. Unlike `model_parts.SincnetConv` (kernel on H), `kernel_size` is honored as-is so Axon's max filter dim of 16 stays respected.

In [ ]:
from model_parts import SincnetConvW

## Build the end-to-end training model

**Stage 1 — per-chunk frontend** (one model in the graph, replicated implicitly across the 46 chunk-rows on H):
`SincnetConvW(k=15, s=8, 32 filters)` → ReLU → `Conv2D(1,5) same, 20 filters` → ReLU → `MaxPool(1,16)` → `MaxPool(1,7)`.
Output: `(B, 46, 1, 20)` — **20 features per chunk × 46 chunks = 920 ≤ 1024** (fits the user's stage-1 output budget).

**Stage 2 — hierarchical head over the chunk sequence** (mimics `sincnet_multi`):
`Conv2D(5,1) same, 32 filters` → ReLU → `AvgPool(2,1)` → `Conv2D(5,1) same, 32 filters` → ReLU → `AvgPool(23,1)` → Flatten → `Dense(64, relu)` → `Dense(num_labels)`.

In [ ]:
from utils import get_flops_native


def build_training_model(num_labels: int) -> tf.keras.Model:
    # Match the standard time-domain dataset shape (B, TARGET_AUDIO_LEN_TIME, 1).
    inputs = tf.keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1))

    # Crop the tail to TOTAL_LEN = NUM_CHUNKS * CHUNK_SIZE, then reshape into chunks.
    tail_crop = TARGET_AUDIO_LEN_TIME - TOTAL_LEN
    x = tf.keras.layers.Cropping1D(cropping=(0, tail_crop), name="crop_to_chunks")(inputs)
    x = tf.keras.layers.Reshape((NUM_CHUNKS, CHUNK_SIZE, 1), name="chunkify")(x)

    # === Stage 1: per-chunk frontend ===========================================
    x = SincnetConvW(
        num_filters=SINCNET_FILTERS,
        kernel_size=SINCNET_KERNEL,
        stride=SINCNET_STRIDE,
        sample_rate=SAMPLE_RATE,
        name="sincnet_w",
    )(x)
    x = tf.keras.layers.ReLU(name="sinc_relu")(x)
    # Standard conv composes the sinc filterbank into per-chunk learned features.
    x = tf.keras.layers.Conv2D(
        filters=CHUNK_CONV_FILTERS,
        kernel_size=(1, CHUNK_CONV_KERNEL),
        strides=(1, 1),
        padding="same",
        name="chunk_conv",
    )(x)
    x = tf.keras.layers.ReLU(name="chunk_relu")(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(1, CHUNK_POOL_1), strides=(1, CHUNK_POOL_1),
        padding="valid", name="chunk_pool_1",
    )(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(1, CHUNK_POOL_2), strides=(1, CHUNK_POOL_2),
        padding="valid", name="chunk_pool_2",
    )(x)
    # Shape here: (B, 46, 1, 20) — 20 features per chunk.

    # === Stage 2: hierarchical head over the chunk sequence ====================
    x = tf.keras.layers.Conv2D(
        filters=HEAD_CONV_1_FILTERS,
        kernel_size=(HEAD_CONV_KERNEL, 1),
        strides=(1, 1),
        padding="same",
        name="head_conv_1",
    )(x)
    x = tf.keras.layers.ReLU(name="head_relu_1")(x)
    x = tf.keras.layers.AveragePooling2D(
        pool_size=(HEAD_POOL_1, 1), strides=(HEAD_POOL_1, 1),
        padding="valid", name="head_pool_1",
    )(x)
    x = tf.keras.layers.Conv2D(
        filters=HEAD_CONV_2_FILTERS,
        kernel_size=(HEAD_CONV_KERNEL, 1),
        strides=(1, 1),
        padding="same",
        name="head_conv_2",
    )(x)
    x = tf.keras.layers.ReLU(name="head_relu_2")(x)
    x = tf.keras.layers.AveragePooling2D(
        pool_size=(HEAD_POOL_2, 1), strides=(HEAD_POOL_2, 1),
        padding="valid", name="head_pool_2",
    )(x)
    x = tf.keras.layers.Flatten(name="flatten")(x)
    x = tf.keras.layers.Dense(DENSE_HIDDEN, activation="relu", name="dense_hidden")(x)
    outputs = tf.keras.layers.Dense(num_labels, activation=None, name="dense_logits")(x)

    return tf.keras.Model(inputs, outputs, name="sincnet_chunked_training")


training_model = build_training_model(num_labels)
training_model.summary()

flops = get_flops_native(training_model, batch_size=1)
print(f"Total FLOPs: {flops}")

In [ ]:
from utils import init_wandb, get_callbacks, finish_wandb

training_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

init_wandb(MODEL_STEM, config={
    "chunk_size": CHUNK_SIZE,
    "num_chunks": NUM_CHUNKS,
    "sincnet_filters": SINCNET_FILTERS,
    "sincnet_kernel": SINCNET_KERNEL,
    "sincnet_stride": SINCNET_STRIDE,
    "chunk_conv_filters": CHUNK_CONV_FILTERS,
    "chunk_conv_kernel": CHUNK_CONV_KERNEL,
    "head_conv_1_filters": HEAD_CONV_1_FILTERS,
    "head_conv_2_filters": HEAD_CONV_2_FILTERS,
    "head_conv_kernel": HEAD_CONV_KERNEL,
    "dense_hidden": DENSE_HIDDEN,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
})

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=get_callbacks(10, 5, BATCH_SIZE),
)
finish_wandb()

In [ ]:
from utils import plot_training_history
plot_training_history(history)

## Bake learned sinc filters into a static Conv2D

Replace the dynamic `SincnetConvW` with a frozen `Conv2D` of shape `(1, k, 1, num_filters)` so the int8 TFLite converter sees only standard ops.

In [ ]:
frontend_layer = training_model.get_layer("sincnet_w")
baked_conv = frontend_layer.export_to_conv2d(name="baked_sinc_conv_w")

# Deploy model: skip Cropping1D + Reshape entirely so the Axon compiler sees a
# clean rank-4 NHWC input (B, NUM_CHUNKS, CHUNK_SIZE, 1). On device, the C code
# fills this buffer with raw audio (same memory as a flat NUM_CHUNKS*CHUNK_SIZE
# array — no reshape needed at runtime).
deploy_skip = {"crop_to_chunks", "chunkify", "sincnet_w"}

infer_inputs = tf.keras.Input(shape=(NUM_CHUNKS, CHUNK_SIZE, 1))
x = baked_conv(infer_inputs)
for layer in training_model.layers:
    if isinstance(layer, tf.keras.layers.InputLayer):
        continue
    if layer.name in deploy_skip:
        continue
    x = layer(x)

inference_model = tf.keras.Model(infer_inputs, x, name="sincnet_chunked_inference")

# Numerical check: training_model takes raw audio (B, 47872, 1); inference_model
# takes the already-chunked (B, 46, 1024, 1). Crop the raw input to TOTAL_LEN
# then reshape to match before comparing.
for batch_audio, _ in test_ds.take(1):
    batch_np = batch_audio.numpy()
    batch_chunked = batch_np[:, :TOTAL_LEN, :].reshape(-1, NUM_CHUNKS, CHUNK_SIZE, 1)
    logits_train = training_model.predict(batch_np, verbose=0)
    logits_infer = inference_model.predict(batch_chunked, verbose=0)
    print(f"Max abs diff: {np.max(np.abs(logits_train - logits_infer)):.8f}")

## Quantize and export INT8 TFLite

Representative samples are the same chunked tensors the model expects.

In [ ]:
# Representative batches: the deploy model takes (B, NUM_CHUNKS, CHUNK_SIZE, 1),
# so crop the raw audio to TOTAL_LEN and reshape into chunks for the calibrator.
raw_rep_batches = build_representative_batches(test_ds, take=100)
rep_batches = [
    b[:, :TOTAL_LEN, :].reshape(-1, NUM_CHUNKS, CHUNK_SIZE, 1)
    for b in raw_rep_batches
]

try:
    export_keras_model_to_int8_tflite(inference_model, rep_batches, OUT_TFLITE)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"TFLite conversion failed: {e}")

In [ ]:
from utils import evaluate_tflite_model

hyperparams = {
    "chunk_size": CHUNK_SIZE,
    "num_chunks": NUM_CHUNKS,
    "sincnet_filters": SINCNET_FILTERS,
    "sincnet_kernel": SINCNET_KERNEL,
    "sincnet_stride": SINCNET_STRIDE,
    "chunk_conv_filters": CHUNK_CONV_FILTERS,
    "chunk_conv_kernel": CHUNK_CONV_KERNEL,
    "head_conv_1_filters": HEAD_CONV_1_FILTERS,
    "head_conv_2_filters": HEAD_CONV_2_FILTERS,
    "head_conv_kernel": HEAD_CONV_KERNEL,
    "dense_hidden": DENSE_HIDDEN,
    "target_audio_len": TOTAL_LEN,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds, hyperparams=hyperparams,
)
print(f"Avg inference: {avg_ms:.3f} ms")